In [ ]:
#Already imported necessary libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# Load dataset
df = pd.read_csv("diabities_data.csv")

# Separate features and target
X = df.drop("diabetes", axis=1)
y = df["diabetes"]

# Replace medically implausible zero values with NaN
zero_columns = [
    "glucose",
    "diastolic",
    "triceps",
    "insulin",
    "bmi"
]

for column in zero_columns:
    X[column] = X[column].replace(0, np.nan)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ML pipeline
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train model
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7077922077922078
ROC-AUC: 0.812962962962963

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.82      0.78       100
           1       0.60      0.50      0.55        54

    accuracy                           0.71       154
   macro avg       0.68      0.66      0.67       154
weighted avg       0.70      0.71      0.70       154


Confusion Matrix:
[[82 18]
 [27 27]]


In [2]:
# New patient diabetic prediction
new_patient = pd.DataFrame({
    "pregnancies": [2],
    "glucose": [140],
    "diastolic": [70],
    "triceps": [30],
    "insulin": [100],
    "bmi": [32.0],
    "dpf": [0.5],
    "age": [35]
})

prediction = model.predict(new_patient)[0]
probability = model.predict_proba(new_patient)[0][1]

if prediction == 1:
    print("Prediction: Diabetic")
else:
    print("Prediction: Non-Diabetic")

print(f"Predicted probability: {probability:.2%}")

Prediction: Non-Diabetic
Predicted probability: 41.99%
